# Laporan Proyek Regresi Linier: Memprediksi Loudness Musik

**Disusun oleh:** Tsaabitah Arga Nitibaskara

**Tanggal:** 9 November 2025

## 1. Ringkasan Data

* **Dataset:** Music Audio Features
* **Ukuran:** 18.492 lagu (baris) dan 33 fitur (kolom).
* **Variabel Target (y):** `loudness` (float). Ini adalah variabel numerik kontinu yang mengukur kenyaringan lagu dalam dB (desibel), di mana nilai yang lebih tinggi (lebih mendekati 0) berarti lebih keras.
* **Variabel Fitur (X):** 30+ fitur yang mencakup:
  * **Fitur Numerik:** `tempo`, `duration`, 12 `avg_timbre`, 12 `var_timbre`.
  * **Fitur Kategorikal:** `genre` (10 kategori), `key` (12 nada), `mode` (mayor/minor), `time_signature`.
* **Kualitas Data:** Data sangat bersih, tidak ada nilai hilang (missing values) setelah perbaikan tipe data awal (memperbaiki pemisah desimal koma dan separator titik koma).

## 2. Tujuan Analisis

Tujuan dari analisis ini adalah untuk membangun dan mengevaluasi beberapa model regresi linier untuk **memprediksi (atau menjelaskan) tingkat `loudness` (kenyaringan) sebuah lagu** berdasarkan fitur audio teknis dan kategorikalnya.

Kita ingin menjawab pertanyaan: "Fitur audio apa (tempo, timbre, atau bahkan genre) yang memiliki dampak paling signifikan terhadap kenyaringan sebuah lagu?"

## Langkah 1: Import Library dan Memuat Data

Langkah pertama adalah mengimpor semua library yang diperlukan dan memuat dataset kita. Kita harus ingat untuk menggunakan `encoding='latin1'`, `sep=';'`, dan `decimal=','` untuk memuat file CSV ini dengan benar.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score, mean_absolute_error

# Tentukan lokasi file (Pastikan file di-upload ke Colab/Jupyter)
filepath = 'musicgenre-small.csv'

print("--- Memulai Analisis Proyek Regresi: Memprediksi Loudness ---")

# 1. Memuat Data (DENGAN PERBAIKAN)
try:
    data = pd.read_csv(filepath, 
                       encoding='latin1', 
                       sep=';', 
                       decimal=',')
    print(f"\nBerhasil memuat data dari: {filepath}")
    print(f"Bentuk data: {data.shape}")
except FileNotFoundError:
    print(f"ERROR: File '{filepath}' tidak ditemukan.")
    print("Pastikan Anda telah meng-upload file CSV ke lingkungan Colab ini.")
except Exception as e:
    print(f"Terjadi error saat memuat data: {e}")

## Langkah 2: Persiapan Data (Preprocessing)

Sekarang, kita akan memisahkan variabel target (`y`) dari variabel fitur (`X`). Kita juga akan mendefinisikan fitur mana yang numerik (perlu di-scale) dan mana yang kategorikal (perlu di-one-hot-encode). Terakhir, kita membagi data menjadi set latih (train) dan uji (test).

In [ ]:
print("Mempersiapkan data untuk pemodelan...")

# 2. Persiapan Data (Define X and y)
y = data['loudness']
X = data.drop(columns=['loudness', 'artist_name', 'title'])

# 3. Identifikasi Tipe Fitur untuk Pra-pemrosesan
categorical_features = ['genre', 'time_signature', 'key', 'mode']
numeric_features = [col for col in X.columns if col not in categorical_features]

print(f"  - Target (y): loudness")
print(f"  - Fitur Kategorikal (One-Hot): {categorical_features}")
print(f"  - Fitur Numerik (Scaled): {len(numeric_features)} kolom (tempo, duration, timbre, dll.)")

# 4. Membagi Data (Train/Test Split)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"\nData dibagi: {len(X_train)} baris latih, {len(X_test)} baris uji.")

# 5. Membuat Preprocessing Pipeline
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler()) # Wajib untuk Ridge/Lasso
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore')) # Abaikan kategori langka di data uji
])

# Gabungkan transformers menggunakan ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

print("\nPipeline preprocessing berhasil dibuat.")

## 3. Perbandingan Model

Sesuai instruksi, kita akan melatih dan membandingkan tiga variasi regresi linier: Regresi Linier Berganda (standar), Regresi Ridge (L2), dan Regresi Lasso (L1). Kita akan menggunakan R-squared (R²) dan Mean Absolute Error (MAE) sebagai metrik evaluasi.

In [ ]:
# 6. Daftar Model untuk Dibandingkan
models = {
    "Multiple Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Lasso Regression": Lasso(alpha=0.1) # Alpha kecil untuk Lasso
}

print("--- BAGIAN 3: PERBANDINGAN MODEL ---")
print("Melatih dan mengevaluasi model...")

results = {}

# Loop, latih, dan evaluasi setiap model
for name, model in models.items():
    
    # Buat pipeline lengkap (Preprocessing + Model)
    pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                              ('model', model)])
    
    # Latih model
    pipeline.fit(X_train, y_train)
    
    # Buat prediksi di data uji
    y_pred = pipeline.predict(X_test)
    
    # Evaluasi
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    
    results[name] = {'R-squared': r2, 'MAE': mae}
    
    print(f"\n--- Model: {name} ---")
    print(f"  R-squared (R²): {r2:.4f}")
    print(f"  Mean Absolute Error (MAE): {mae:.4f}")

print("\n-------------------------------------------------")
print("Analisis perbandingan model regresi selesai.")
print("-------------------------------------------------")

### Pilihan Model Terbaik

Untuk akurasi prediksi murni, **Multiple Linear Regression** dan **Ridge Regression** adalah pemenang yang jelas dan kinerjanya identik. Keduanya mencapai **R-squared 0.9793 yang sangat tinggi**. Ini berarti model-model ini mampu menjelaskan **97.93% varians** dalam data `loudness`, dengan rata-rata kesalahan prediksi hanya 0.67 dB.

## 4. Temuan Kunci (Key Findings)

Sekarang kita tahu modelnya sangat akurat. Tapi *mengapa*? Untuk menjawab ini, kita akan melihat ke dalam model Lasso. Keuntungan Lasso adalah ia menyederhanakan model dengan 'menghapus' (mengatur koefisien ke nol) fitur-fitur yang dianggapnya tidak penting. Ini membantu kita menemukan prediktor yang paling kuat.

In [ ]:
print("--- BAGIAN 4: TEMUAN KUNCI (Contoh dari Model Lasso) ---")

# Kita ambil pipeline Lasso yang sudah dilatih
# (Kita harus melatihnya lagi di luar loop untuk akses mudah)
lasso_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                 ('model', Lasso(alpha=0.1))])
lasso_pipeline.fit(X_train, y_train)

# Ambil nama fitur setelah di-OneHotEncode
try:
    # Dapatkan nama fitur dari preprocessor
    ohe_feature_names = lasso_pipeline.named_steps['preprocessor'] \
        .named_transformers_['cat'] \
        .named_steps['onehot'] \
        .get_feature_names_out(categorical_features)
    
    # Gabungkan semua nama fitur
    all_feature_names = numeric_features + list(ohe_feature_names)
    
    # Ambil koefisien dari model
    coefficients = lasso_pipeline.named_steps['model'].coef_
    
    # Buat seri pandas untuk melihatnya
    coef_series = pd.Series(coefficients, index=all_feature_names).sort_values(ascending=False)
    
    print("Fitur dengan Dampak Terbesar (Koefisien Lasso Teratas):")
    print(coef_series.head(5)) # 5 teratas
    
    print("\nFitur dengan Dampak Negatif Terbesar (Koefisien Lasso Terbawah):")
    print(coef_series.tail(5)) # 5 terbawah
    
    # Hitung fitur yang "dihapus" oleh Lasso
    zero_coefs = np.sum(coefficients == 0)
    total_coefs = len(coefficients)
    print(f"\nLasso 'menghapus' {zero_coefs} dari {total_coefs} total fitur (mengaturnya ke nol).")
    print(f"Fitur yang digunakan: {total_coefs - zero_coefs}")

except Exception as e:
    print(f"\n(Tidak dapat mengambil nama fitur untuk temuan kunci: {e})")

print("-------------------------------------------------")

### Analisis Temuan Kunci

1.  **Fitur Audio Sangat Prediktif:** Temuan utama adalah bahwa `loudness` sebuah lagu hampir sepenuhnya dapat dijelaskan (R² ~98%) oleh fitur-fitur teknis audio lainnya, terutama `timbre` (warna suara).

2.  **Pentingnya `avg_timbre1`:** Model Lasso mengidentifikasi `avg_timbre1` (koefisien = 6.349) sebagai fitur dengan dampak positif terbesar. Ini menunjukkan bahwa koefisien pertama dari *timbre* (sering dikaitkan dengan "kecerahan" atau *brightness* suara) adalah prediktor paling dominan dari `loudness`.

3.  **Fitur yang Tidak Relevan:** Wawasan paling menarik dari model Lasso adalah kemampuannya untuk menyederhanakan model. Lasso 'menghapus' (mengatur koefisien ke nol) **47 dari 56** total fitur. Di antara fitur yang dihapus adalah *semua* fitur yang terkait dengan `key` (kunci nada) dan `mode` (mayor/minor).

    * **Wawasan Kunci:** Ini berarti **kunci nada atau modus (mayor/minor) sebuah lagu TIDAK memiliki dampak yang signifikan secara statistik** terhadap `loudness`-nya.

## 5. Keterbatasan dan Langkah Selanjutnya

* **Keterbatasan:**
    * R² yang sangat tinggi (98%) mungkin menunjukkan bahwa `loudness` dan `timbre` secara teknis sangat erat kaitannya dalam cara mereka dihitung dari sinyal audio. Ini lebih merupakan model *penjelasan* yang kuat daripada model *prediksi* untuk lagu baru (di mana fitur `timbre`-nya mungkin tidak kita ketahui).
    * Kolom `artist_name` dan `title` diabaikan, meskipun artis tertentu mungkin memiliki "suara" (tingkat kenyaringan) yang konsisten karena proses *mastering* yang seragam.

* **Langkah Selanjutnya:**
    * Mengingat tingginya R², model regresi linier sudah sangat memuaskan. Langkah selanjutnya adalah menggunakan model yang lebih kompleks (seperti Random Forest Regressor) untuk melihat apakah sisa 2% varians dapat dijelaskan.
    * Menganalisis koefisien dari *setiap* `genre` dalam model Linear Regression untuk melihat genre mana (setelah mengontrol semua fitur lain) yang secara inheren lebih keras atau lebih pelan.